# Description
In this module, the prediction error dictionary is formed based on the arrival time predictions computed in the previous module. 

The dictionary contains the actual time of arrival, predicted arrival time, error in arrival time prediction for all the bus-stops of the routes. 

In [1]:
import json
from pymongo import MongoClient
import math
import numpy as np

In [2]:
import pprint

In [3]:
con = MongoClient()

In [4]:
RouteName='Git_ISCON_PDPU'

In [5]:
def TActual(TripIndex, SingleTripsInfo, BusStopsCount, TripStartHour, Bound,
            RouteName, ResultPathDir, ResultPathDir_Np, UseMongoDB, PredictionDictLists):
    '''
    Input: Trip index, trips list, bus-stop count, trip start hour, trip direction (or bound)
    route name, directory paths, the flag `UseMongoDB` to determine whether MongoDB database or
    numpy files are used, and dictionary of prediction and actual values.
    Output: The actual time for different bus-stop for the provided trip
    Function: It extracts the actual time at different bus-stops for the provided trip.
    '''
    TActualNpArray =  np.full((BusStopsCount),np.inf,dtype=float)
    for index in range(BusStopsCount):
        
        AcutalValueList = [record['TActual'] for record in PredictionDictLists
                           if record['id']==index]
        
        if len(AcutalValueList)!=0:
            try:
                TActualNpArray[index] = AcutalValueList[0]
                #print('AcutalValueList', AcutalValueList)
            except IndexError:
                print('AcutalValueList Error', AcutalValueList)
                #input()
    return(TActualNpArray)

In [6]:
def ErrorMatrix(TripIndex, SingleTripsInfo, BusStopsCount, TripStartHour, Bound,
                RouteName, ResultPathDir, ResultPathDir_Np, UseMongoDB):

    '''
    Input: Trip index, trips list, bus-stop count, trip start hour, trip direction (or bound)
    route name, directory paths, the flag `UseMongoDB` to determine whether MongoDB database or
    numpy files are used.
    Output: The dictionary of prediction error for different bus-stop for the provided trip.
    Function: It extracts the prediction value and computes the dictionary of prediction error for 
    different bus-stop for the provided trip.
    '''    
    
    SingleTripInfo = SingleTripsInfo[TripIndex]
    
    if UseMongoDB==True:
        PredictionDictLists = [record for record in 
                               con[RouteName][SingleTripsInfo[TripIndex]+'.PredictionResult_Dist_th_50'].find()]
    
    else:
        PredictionDictLists = np.load(f'{ResultPathDir_Np}/{RouteName}/{SingleTripInfo}.PredictionResult_Dist_th_50.npy', allow_pickle=True)
        
    TActualNpArray = TActual(TripIndex, SingleTripsInfo, BusStopsCount, TripStartHour, Bound,
                             RouteName, ResultPathDir, ResultPathDir_Np, UseMongoDB, PredictionDictLists)

    
    
    ErrorNpMatrix = np.full((BusStopsCount,BusStopsCount),np.inf,dtype=float)

    for index in range (BusStopsCount):
        PredictionDictList = [rec for rec in PredictionDictLists if rec['id']==index]
            
        if len(PredictionDictList)!=0:
            if PredictionDictList[0]['PredictionAvailable'] ==True:
                for PredIndex in range(len(PredictionDictList[0]['PredictionTupleList'])):
                    BusStopPredictionIndex = PredictionDictList[0]['PredictionTupleList'][PredIndex][0]
                    if TActualNpArray[BusStopPredictionIndex]!=np.inf:
                        
                        '''For margin considering in prediction error calculation'''
                        ErrorList =[]
                        ErrorList.append(abs(TActualNpArray[BusStopPredictionIndex]-PredictionDictList[0]['PredictionTupleList'][PredIndex][1]))
                        # The prediction and margin are generated at same time and stored subsequently. Hence PredictionIndex must get access 
                        # for the same ID
                        
                        if PredictionDictList[0]['PredictionMarginTupleList'][PredIndex][0]==PredictionDictList[0]['PredictionTupleList'][PredIndex][0]:
                            PredictionL=int(PredictionDictList[0]['PredictionTupleList'][PredIndex][1]-PredictionDictList[0]['PredictionMarginTupleList'][PredIndex][1])
                            PredictionH=int(PredictionDictList[0]['PredictionTupleList'][PredIndex][1]+PredictionDictList[0]['PredictionMarginTupleList'][PredIndex][1])
                            for Prediction in range (PredictionL,PredictionH,1000):
                                ErrorList.append(abs(TActualNpArray[BusStopPredictionIndex]-Prediction))
                            #ErrorList.append(abs(TActualNpArray[BusStopPredictionIndex]-()))
                            #ErrorList.append(abs(TActualNpArray[BusStopPredictionIndex]-(PredictionDictList[0]['PredictionTupleList'][PredIndex][1]+PredictionDictList[0]['PredictionMarginTupleList'][PredIndex][1])))
                            
                        ErrorConsideringMargin=min(ErrorList)
                        ErrorNpMatrix[PredictionDictList[0]['PredictionTupleList'][PredIndex][0]][index]=ErrorConsideringMargin
                        '''For margin considering in prediction error calculation'''
                        
                        #ErrorNpMatrix[PredictionDictList[0]['PredictionTupleList'][PredIndex][0]][index]=abs(TActualNpArray[BusStopPredictionIndex]-PredictionDictList[0]['PredictionTupleList'][PredIndex][1])
                        
    return(ErrorNpMatrix)                        

In [7]:
def InitializeErrorMatrixList(SingleTripsInfo, BusStopsCount, TripStartHour, Bound,
                              RouteName, ResultPathDir, ResultPathDir_Np, UseMongoDB):
    '''
    Input: Trips list, bus-stop count, trip start hour, trip direction (or bound)
    route name, directory paths, the flag `UseMongoDB` to determine whether MongoDB database or
    numpy files are used, and dictionary of prediction and actual values.
    Output: The dictionary of prediction error for different bus-stop for all the trips.
    Function: It forms the dictionary of prediction error for different bus-stop for all the trips.
    '''    
    ErrorMatrixList = []
    for TripIndex in range(len(SingleTripsInfo)):
        ErrorMatrixList.append(ErrorMatrix(TripIndex, SingleTripsInfo, BusStopsCount, TripStartHour, Bound,
                                           RouteName, ResultPathDir, ResultPathDir_Np, UseMongoDB))
    return(ErrorMatrixList)


In [8]:
def GetErrorMatrixList(SingleTripsInfo, BusStopsCount, TripStartHour, Bound,
                       RouteName, ResultPathDir, ResultPathDir_Np, UseMongoDB
                      ):
    
    '''
    Input: Trips list, bus-stop count, trip start hour, trip direction (or bound)
    route name, directory paths, the flag `UseMongoDB` to determine whether MongoDB database or
    numpy files are used.
    Output: None
    Function: It extracts the prediction value and computes the dictionary of prediction error for 
    different bus-stop and stores either in MongoDB or in Numpy file.
    '''    
    ErrorMatrixList = InitializeErrorMatrixList(SingleTripsInfo, BusStopsCount, TripStartHour, Bound,
                                                RouteName, ResultPathDir, ResultPathDir_Np, UseMongoDB)

    ErrorDictList =[]
    for PredictionForIndex in range(BusStopsCount):
        ErrorDict ={}
        ErrorDict['id']=PredictionForIndex
        ErrorDict['PredictionByIndexList']=[]
        PredictionAvailable = False

        '''List for Error for given BusStop by all stops in all trips'''
        ErrorListForID =[]

        for PredictionByIndex in range(BusStopsCount):
            '''List for Error for given BusStop by given stop in all trips'''
            ErrorList =[]

            for ErrorIndex in range(len(ErrorMatrixList)):
                if ErrorMatrixList[ErrorIndex][PredictionForIndex][PredictionByIndex] != np.inf:
                    ErrorList.append(ErrorMatrixList[ErrorIndex][PredictionForIndex][PredictionByIndex])

                    ErrorListForID.append(ErrorMatrixList[ErrorIndex][PredictionForIndex][PredictionByIndex])
            if len(ErrorList) != 0:
                PredictionAvailable = True
                ErrorListNP = np.asarray(ErrorList)
                print('PredictionFor: '+str(PredictionForIndex))
                print('PredictionByIndex: '+str(PredictionByIndex))
                print(ErrorList)
                #input()
                ErrorDict['PredictionByIndexList'].append(PredictionByIndex)
                ErrorDict[str(PredictionByIndex)]=(np.mean(ErrorListNP),np.std(ErrorListNP))

        if len(ErrorListForID)!=0:
            ErrorListForIDNP = np.asarray(ErrorListForID)

            ErrorDict['PredictionErrorAggregateMean'] = np.mean(ErrorListForIDNP)
            ErrorDict['PredictionErrorAggregateSTD'] = np.std(ErrorListForIDNP)
            ErrorDict['PredictionErrorAggregateMax'] = np.amax(ErrorListForIDNP)

            ErrorListForID.sort()
            NinetyPercentileValueIndex = int(len(ErrorListForID)*0.9)

            ErrorDict['PredictionErrorAggregateNinetyPercentileValue'] = ErrorListForID [NinetyPercentileValueIndex]

        ErrorDict['PredictionAvailable'] = PredictionAvailable
        ErrorDictList.append(ErrorDict)
    
    print('ErrorDictList', ErrorDictList)
    
    if UseMongoDB==True:
        #con[RouteName].drop_collection(f'PredictionErrorForSubStopsV2.{TripStartHour}.{Bound}')
        con[RouteName][f'PredictionErrorForSubStopsV2.{TripStartHour}.{Bound}'].insert_many(ErrorDictList)
    else:
        np.save(f'{ResultPathDir_Np}/{RouteName}/PredictionErrorForSubStopsV2.{TripStartHour}.{Bound}.npy',
                ErrorDictList
               )
        
    return(ErrorDictList)

## Considered Trips for prediction

In [9]:
from pathlib import Path
import os
'''For directory management'''

path = Path(os.getcwd())

OneLevelUpPath = path.parents[0]
NpPathDir = os.path.join(str(OneLevelUpPath), 'data','NpData')
ResultPathDir = os.path.join(str(OneLevelUpPath), 'results','PredictionError','')

ResultPathDir_Np = os.path.join(str(OneLevelUpPath), 'results','NpData','')

In [10]:
#'''
ProjectDataUsed = True
UsedPreTrained = False
UseMongoDB = True
#'''
'''
ProjectDataUsed = True
UsedPreTrained = True
UseMongoDB = False
'''

'\nProjectDataUsed = True\nUsedPreTrained = True\nUseMongoDB = False\n'

'''SingleTripsInfo is initialized with list of selected trips '''

In [11]:
SingleTripsInfo = ['19_01_2018__07_38_47',
 '22_12_2017__07_38_21',
 '20_12_2017__07_38_14',
 '29_12_2017__07_37_27',
 '29_01_2018__07_39_47',
 '30_01_2018__07_42_30',
 '02_02_2018__07_38_50',
 '12_02_2018__07_40_14',
 '16_02_2018__07_45_41',
# '14_03_2018__07_35_46',
# '20_03_2018__07_28_45',
# '22_03_2018__07_38_43',
 '14_02_2018__07_41_04',
 '22_02_2018__07_42_45',
# '03_04_2018__07_38_31',
 #Added morning
'18_01_2018__07_38_10',
'08_01_2018__07_41_43',
'09_01_2018__07_40_01',]

In [12]:
if UseMongoDB==True:
    TripInfo = [rec for rec in con[RouteName]['TripInfo'].find({'SingleTripInfo':SingleTripsInfo[0]})]
    TripStartHour = TripInfo[0]['TripStartHour']
    Bound = TripInfo[0]['Bound']
    
    BusStopsList = [BusStop for BusStop in con[RouteName][f'BusStops.{Bound}Bound'].find().sort([('id',1)])]
else:
    TripsInfo = np.load(f'{NpPathDir}/{RouteName}/TripInfo.npy',allow_pickle=True)
    TripInfo = [rec for rec in TripsInfo if rec['SingleTripInfo']==SingleTripsInfo[0]]
    TripStartHour = TripInfo[0]['TripStartHour']
    Bound = TripInfo[0]['Bound']

    BusStopsList = np.load(f'{NpPathDir}/{RouteName}/BusStops.{Bound}Bound.npy',allow_pickle=True)
    
print('StartHour, Bound, BusStopsCount', TripStartHour, Bound, len (BusStopsList))

StartHour, Bound, BusStopsCount 07 North 15


In [13]:
BusStopsCount = len (BusStopsList)

In [14]:
ErrorDictList = GetErrorMatrixList(SingleTripsInfo, BusStopsCount, TripStartHour, Bound,
                                   RouteName, ResultPathDir, ResultPathDir_Np, UseMongoDB
                                  )

PredictionFor: 1
PredictionByIndex: 1
[np.float64(64000.0)]
PredictionFor: 2
PredictionByIndex: 1
[np.float64(19000.0)]
PredictionFor: 2
PredictionByIndex: 2
[np.float64(11832.0), np.float64(218.0), np.float64(93.0), np.float64(179950.0), np.float64(70561.0), np.float64(10193.0), np.float64(247.0), np.float64(170.0), np.float64(48.0), np.float64(7085.0), np.float64(79683.87084960938), np.float64(471.0), np.float64(15583.0)]
PredictionFor: 3
PredictionByIndex: 1
[np.float64(54740.0)]
PredictionFor: 3
PredictionByIndex: 2
[np.float64(37837.0), np.float64(6634.0), np.float64(224.0), np.float64(162015.0), np.float64(90996.0), np.float64(71078.0), np.float64(2435.0), np.float64(22.0), np.float64(41013.0), np.float64(160817.0), np.float64(2227.0), np.float64(26766.0)]
PredictionFor: 3
PredictionByIndex: 3
[np.float64(463.0), np.float64(99.0), np.float64(106.0), np.float64(180.0), np.float64(48114.0), np.float64(276.0), np.float64(9289.0), np.float64(21521.0), np.float64(20378.0), np.float64(

In [15]:
pprint.pprint(ErrorDictList)

[{'PredictionAvailable': False,
  'PredictionByIndexList': [],
  '_id': ObjectId('6ab4bf82ddb0d8c36141dab0'),
  'id': 0},
 {'1': (np.float64(64000.0), np.float64(0.0)),
  'PredictionAvailable': True,
  'PredictionByIndexList': [1],
  'PredictionErrorAggregateMax': np.float64(64000.0),
  'PredictionErrorAggregateMean': np.float64(64000.0),
  'PredictionErrorAggregateNinetyPercentileValue': np.float64(64000.0),
  'PredictionErrorAggregateSTD': np.float64(0.0),
  '_id': ObjectId('6ab4bf82ddb0d8c36141dab1'),
  'id': 1},
 {'1': (np.float64(19000.0), np.float64(0.0)),
  '2': (np.float64(28933.451603816105), np.float64(50660.85366764)),
  'PredictionAvailable': True,
  'PredictionByIndexList': [1, 2],
  'PredictionErrorAggregateMax': np.float64(179950.0),
  'PredictionErrorAggregateMean': np.float64(28223.91934640067),
  'PredictionErrorAggregateNinetyPercentileValue': np.float64(79683.87084960938),
  'PredictionErrorAggregateSTD': np.float64(48885.00543021303),
  '_id': ObjectId('6ab4bf82ddb

In [16]:
#SingleTripsInfo=[Trip['SingleTripInfo'] for Trip in con[RouteName]['TripInfo'].find({'BusStopRecordExtracted':True,'Bound':'South','TripStartHour':'18'})]

SingleTripsInfo=[
 '22_12_2017__18_38_34',
 '20_12_2017__18_31_19',
 '08_01_2018__18_37_49',
 '14_02_2018__18_30_22',
 '15_02_2018__18_33_19',
 #'03_04_2018__18_32_45',
 
 '20_02_2018__18_31_07',
 '21_02_2018__18_28_29',
 '28_03_2018__18_30_02',
 '04_04_2018__18_34_54'
]

In [17]:
if UseMongoDB==True:
    TripInfo = [rec for rec in con[RouteName]['TripInfo'].find({'SingleTripInfo':SingleTripsInfo[0]})]
    TripStartHour = TripInfo[0]['TripStartHour']
    Bound = TripInfo[0]['Bound']
    
    BusStopsList = [BusStop for BusStop in con[RouteName][f'BusStops.{Bound}Bound'].find().sort([('id',1)])]
else:
    TripsInfo = np.load(f'{NpPathDir}/{RouteName}/TripInfo.npy',allow_pickle=True)
    TripInfo = [rec for rec in TripsInfo if rec['SingleTripInfo']==SingleTripsInfo[0]]
    TripStartHour = TripInfo[0]['TripStartHour']
    Bound = TripInfo[0]['Bound']

    BusStopsList = np.load(f'{NpPathDir}/{RouteName}/BusStops.{Bound}Bound.npy',allow_pickle=True)
    
print('StartHour, Bound, BusStopsCount', TripStartHour, Bound, len (BusStopsList))

StartHour, Bound, BusStopsCount 18 South 15


In [18]:
BusStopsCount = len (BusStopsList)

In [19]:
ErrorDictList = GetErrorMatrixList(SingleTripsInfo, BusStopsCount, TripStartHour, Bound,
                                   RouteName, ResultPathDir, ResultPathDir_Np, UseMongoDB
                                  )

PredictionFor: 0
PredictionByIndex: 0
[np.float64(21258.0), np.float64(30909.0), np.float64(254.0), np.float64(234.0), np.float64(293.0), np.float64(36698.0), np.float64(396.0), np.float64(417.0)]
PredictionFor: 0
PredictionByIndex: 1
[np.float64(75255.0), np.float64(483.0), np.float64(401.0), np.float64(268.0), np.float64(105755.0), np.float64(414.0), np.float64(266414.0), np.float64(152.0)]
PredictionFor: 0
PredictionByIndex: 2
[np.float64(66085.0), np.float64(16998.0), np.float64(55560.0), np.float64(95069.0), np.float64(57.0), np.float64(477.0), np.float64(396685.0), np.float64(396.0)]
PredictionFor: 0
PredictionByIndex: 3
[np.float64(75.0), np.float64(920.0), np.float64(15231.0), np.float64(207286.0), np.float64(110858.0), np.float64(229231.0), np.float64(642050.0), np.float64(133927.0)]
PredictionFor: 0
PredictionByIndex: 4
[np.float64(263146.0), np.float64(468332.0), np.float64(172391.0), np.float64(148202.0), np.float64(479.0), np.float64(112.0), np.float64(476583.0), np.float6

In [20]:
pprint.pprint(ErrorDictList)

[{'0': (np.float64(11307.375), np.float64(14712.639293626926)),
  '1': (np.float64(56142.75), np.float64(88464.99861491832)),
  '10': (np.float64(351798.5714285714), np.float64(152852.1052276697)),
  '11': (np.float64(373557.5), np.float64(167109.23586385045)),
  '12': (np.float64(358111.75), np.float64(181866.63190628868)),
  '13': (np.float64(451100.8), np.float64(151968.02713386787)),
  '2': (np.float64(78915.875), np.float64(124658.95565144678)),
  '3': (np.float64(167447.25), np.float64(198082.2620388749)),
  '4': (np.float64(208888.25), np.float64(172643.02133140946)),
  '5': (np.float64(216207.25), np.float64(159406.72984126955)),
  '6': (np.float64(254679.375), np.float64(178123.36641492147)),
  '7': (np.float64(302622.625), np.float64(200190.54996736077)),
  '8': (np.float64(314931.75), np.float64(147599.9984576135)),
  '9': (np.float64(349801.14285714284), np.float64(178151.76465941983)),
  'PredictionAvailable': True,
  'PredictionByIndexList': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9,